#  Student Dropout Prediction — SVM (Support Vector Machine)

**Dataset:** `student_dropout_dataset_v3.csv`  
**Task:** Binary Classification — Prediksi Dropout  
**Model:** Support Vector Classifier (SVC)

---
###  Daftar Isi
1. Cara Melihat Tipe Data
2. Dataset Bisa Digunakan Untuk Apa
3. Model Yang Bisa Digunakan
4. Parameter Yang Bisa Diubah/Disetel
5. Evaluasi Yang Dipakai
6. Cara Mengetahui Evaluasi Bagus atau Tidak
7. Cara Mengoptimasi Model
8. Cara Menyimpan Model
9. Cara Menggunakan Model Hasil Training

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

print('Libraries loaded ')

Libraries loaded 


---
## 1.  Cara Melihat Tipe Data

**Fungsi yang digunakan:**
```python
df.info()          # Ringkasan dataset: kolom, tipe data, non-null count
df.dtypes          # Tipe data per kolom
df.describe()      # Statistik numerik
df.isnull().sum()  # Jumlah missing values
df.nunique()       # Jumlah nilai unik per kolom
```

**SVM sangat sensitif terhadap skala fitur!** Semua fitur numerik HARUS di-scale.

In [2]:
df = pd.read_csv('../student_dropout_dataset_v3.csv')
print('Shape:', df.shape)
df.info()
print('\nMissing values:', df.isnull().sum().sum())

Shape: (10000, 19)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Student_ID             10000 non-null  int64  
 1   Age                    10000 non-null  float64
 2   Gender                 10000 non-null  object 
 3   Family_Income          9500 non-null   float64
 4   Internet_Access        10000 non-null  object 
 5   Study_Hours_per_Day    9500 non-null   float64
 6   Attendance_Rate        10000 non-null  float64
 7   Assignment_Delay_Days  10000 non-null  int64  
 8   Travel_Time_Minutes    10000 non-null  float64
 9   Part_Time_Job          10000 non-null  object 
 10  Scholarship            10000 non-null  object 
 11  Stress_Index           9500 non-null   float64
 12  GPA                    10000 non-null  float64
 13  Semester_GPA           10000 non-null  float64
 14  CGPA                   10000 non-nul

In [3]:
print('Distribusi Target (Dropout):')
print(df['Dropout'].value_counts())
print(f'Imbalance ratio: {(df["Dropout"]==0).sum()}:{(df["Dropout"]==1).sum()}')

Distribusi Target (Dropout):
Dropout
0    7646
1    2354
Name: count, dtype: int64
Imbalance ratio: 7646:2354


---
## 2.  Dataset Bisa Digunakan Untuk Apa

| Tujuan | Target | Jenis |
|--------|--------|-------|
| **Prediksi Dropout** ← (ini) | `Dropout` | Classification |
| Prediksi CGPA | `CGPA` | Regression (SVR) |
| Segmentasi mahasiswa | - | Clustering |
| Analisis faktor | - | EDA |

---
## 3.  Kenapa SVM?

SVM bekerja dengan mencari **hyperplane** yang memaksimalkan margin antara dua kelas:
- **Kernel Trick:** Mengubah data ke dimensi lebih tinggi agar jadi linearly separable
- Sangat efektif untuk data dengan **dimensi tinggi** dan **data yang tidak terlalu besar**
- **Robust** terhadap outlier (hanya mengandalkan support vectors)

| Kernel | Kapan Digunakan |
|--------|----------------|
| `linear` | Data dapat dipisahkan secara linear |
| `rbf` | Data non-linear, paling umum digunakan |
| `poly` | Pola polynomial |
| `sigmoid` | Mirip neural network |

** Catatan:** SVM lambat untuk dataset besar (> 10.000 baris). Pertimbangkan LinearSVC.

In [ ]:
# Preprocessing
df_proc = df.drop(columns=['Student_ID']).copy()
le = LabelEncoder()
for col in df_proc.select_dtypes(include='object').columns:
    df_proc[col] = le.fit_transform(df_proc[col].astype(str))

X = df_proc.drop(columns=['Dropout'])
y = df_proc['Dropout']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Imputasi nilai null dengan median
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

# WAJIB scaling untuk SVM!
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_imp)
X_test_s  = scaler.transform(X_test_imp)

print(f'Train: {X_train_s.shape}, Test: {X_test_s.shape}')

Train: (8000, 17), Test: (2000, 17)


---
## 4.  Parameter Yang Bisa Diubah / Disetel

| Parameter | Default | Penjelasan |
|-----------|---------|------------|
| `C` | 1.0 | Penalty parameter. Besar = fit lebih ketat ke training (risiko overfit). Kecil = lebih smooth (underfit) |
| `kernel` | `'rbf'` | Tipe kernel: `'linear'`, `'rbf'`, `'poly'`, `'sigmoid'` |
| `gamma` | `'scale'` | Pengaruh satu sampel. `'scale'`=1/(n_features*X.std()), `'auto'`=1/n_features. Besar = radius kecil |
| `degree` | 3 | Derajat polynomial (hanya untuk kernel `'poly'`) |
| `class_weight` | None | `'balanced'` untuk imbalanced data |
| `probability` | False | Set `True` untuk bisa gunakan `predict_proba()` |
| `max_iter` | -1 | Batas iterasi solver (-1 = tidak terbatas) |

**Tips umum:**
- Mulai dengan `kernel='rbf'`, `C=1.0`, `gamma='scale'`
- Jika linear separable → coba `kernel='linear'`
- Tune `C` dan `gamma` bersamaan dengan grid search

In [5]:
# Build SVM Model
model = SVC(
    C=1.0,                   # Regularization parameter
    kernel='rbf',            # 'linear', 'rbf', 'poly', 'sigmoid'
    gamma='scale',           # 'scale', 'auto', atau nilai numerik
    class_weight='balanced', # Handle imbalanced class
    probability=True,        # WAJIB True agar bisa pakai predict_proba
    random_state=42
)

model.fit(X_train_s, y_train)
print('SVM berhasil ditraining ')

ValueError: Input X contains NaN.
SVC does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
y_pred = model.predict(X_test_s)
y_pred_proba = model.predict_proba(X_test_s)[:, 1]

---
## 5.  Evaluasi Yang Dipakai

In [ ]:
print('=' * 50)
print(' EVALUASI MODEL SVM')
print('=' * 50)
print(f'Accuracy  : {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision : {precision_score(y_test, y_pred):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred):.4f}')
print(f'F1-Score  : {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC   : {roc_auc_score(y_test, y_pred_proba):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Tidak Dropout', 'Dropout']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Tidak Dropout', 'Dropout'],
            yticklabels=['Tidak Dropout', 'Dropout'], ax=axes[0])
axes[0].set_title('Confusion Matrix - SVM')

from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, 'darkorange', lw=2,
             label=f'AUC={roc_auc_score(y_test, y_pred_proba):.3f}')
axes[1].plot([0,1],[0,1],'navy',linestyle='--')
axes[1].set_title('ROC Curve - SVM')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 6.  Cara Mengetahui Evaluasi Bagus atau Tidak

```
ROC-AUC > 0.85  → Sangat Baik
ROC-AUC > 0.75  → Baik
ROC-AUC > 0.65  → Cukup
ROC-AUC ≈ 0.50  → Tidak lebih baik dari tebak acak
```

**Untuk SVM, cek juga:**
- Apakah jumlah support vectors sangat besar? (> 50% training data) → Model mungkin terlalu lunak
- `model.n_support_` → jumlah support vector per kelas

In [ ]:
train_acc = accuracy_score(y_train, model.predict(X_train_s))
test_acc = accuracy_score(y_test, y_pred)
print(f'Train Accuracy : {train_acc:.4f}')
print(f'Test Accuracy  : {test_acc:.4f}')

print(f'\nJumlah support vectors: {model.n_support_}')
print(f'Total support vectors : {sum(model.n_support_)} dari {len(X_train)} training samples')
pct_sv = sum(model.n_support_) / len(X_train) * 100
print(f'Persentase SV: {pct_sv:.1f}%')
if pct_sv > 50:
    print('  Banyak support vectors (> 50%) — coba C lebih kecil atau data lebih')
else:
    print('  Jumlah support vectors wajar')

---
## 7.  Cara Mengoptimasi Model

| Masalah | Solusi |
|---------|--------|
| **Overfitting** (train >> test) | Kurangi `C`, naikkan regularisasi |
| **Underfitting** | Naikkan `C`, coba kernel berbeda |
| **Lambat** | Gunakan `LinearSVC` atau `SGDClassifier` |
| **Non-linear** | Gunakan `kernel='rbf'` dengan tune `gamma` |
| **Imbalanced** | `class_weight='balanced'` |

**Catatan `C` dan `gamma` untuk RBF:**
- C besar + gamma besar → Overfitting
- C kecil + gamma kecil → Underfitting
- Tuning keduanya penting!

In [ ]:
# Grid Search (lebih kecil jangkauannya karena SVM lambat)
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto', 0.01, 0.1],
    'kernel': ['rbf', 'linear']
}

grid_search = GridSearchCV(
    SVC(class_weight='balanced', probability=True, random_state=42),
    param_grid,
    cv=3,           # Kurangi CV karena SVM lambat
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_s, y_train)
print(' Best params:', grid_search.best_params_)
print(f' Best AUC: {grid_search.best_score_:.4f}')

In [ ]:
best_model = grid_search.best_estimator_
y_b = best_model.predict(X_test_s)
y_bp = best_model.predict_proba(X_test_s)[:, 1]
print(f'Post-Tuning → Acc: {accuracy_score(y_test,y_b):.4f} | F1: {f1_score(y_test,y_b):.4f} | AUC: {roc_auc_score(y_test,y_bp):.4f}')

---
## 8.  Cara Menyimpan Model

Untuk SVM, WAJIB simpan:
1. **Model SVC** — bobot dan support vectors
2. **Scaler** — karena SVM sangat sensitif terhadap skala!

In [ ]:
os.makedirs('saved_models', exist_ok=True)

joblib.dump(best_model, 'saved_models/svm_dropout.pkl')
joblib.dump(scaler, 'saved_models/scaler_svm.pkl')
joblib.dump(list(X.columns), 'saved_models/feature_columns_svm.pkl')

print(' SVM model, scaler, feature columns tersimpan!')
print(f'File size: {os.path.getsize("saved_models/svm_dropout.pkl") / 1024:.1f} KB')

---
## 9.  Cara Menggunakan Model Hasil Training

In [ ]:
loaded_model = joblib.load('saved_models/svm_dropout.pkl')
loaded_scaler = joblib.load('saved_models/scaler_svm.pkl')
loaded_cols = joblib.load('saved_models/feature_columns_svm.pkl')

print('Model dimuat ')

# Data baru
new_data = pd.DataFrame([{
    'Age': 19, 'Gender': 1, 'Family_Income': 22000,
    'Internet_Access': 0, 'Study_Hours_per_Day': 2.0,
    'Attendance_Rate': 62, 'Assignment_Delay_Days': 4,
    'Travel_Time_Minutes': 50, 'Part_Time_Job': 1,
    'Scholarship': 0, 'Stress_Index': 7,
    'GPA': 2.1, 'Semester_GPA': 1.9, 'CGPA': 2.0,
    'Semester': 2, 'Department': 0, 'Parental_Education': 1
}])[loaded_cols]

# INGAT: Harus scale dulu!
new_data_scaled = loaded_scaler.transform(new_data)

pred = loaded_model.predict(new_data_scaled)[0]
prob = loaded_model.predict_proba(new_data_scaled)[0]

print(f'\nPrediksi: {" DROPOUT" if pred == 1 else " TIDAK DROPOUT"}')
print(f'Probabilitas Dropout: {prob[1]:.2%}')
print(f'Probabilitas Tidak Dropout: {prob[0]:.2%}')